In [ ]:
"""
Projeto Final - Ciência de Dados 2026.1 - UFC Sobral
Limpeza e Pré-processamento do Dataset Water Potability
"""
 
import pandas as pd
import numpy as np


# Limpeza do Dataset — Water Potability

Dataset com **3.276 amostras** e **9 features** numéricas contínuas + 1 variável-alvo binária (`Potability`).  
O objetivo desta etapa é tratar inconsistências nos dados brutos antes de qualquer modelagem.

## 1. Carregamento e Inspeção Inicial

Antes de qualquer transformação, é essencial entender o estado bruto dos dados:
quais colunas têm valores ausentes, se existem linhas duplicadas e como está
distribuída a variável-alvo. Essa inspeção guia todas as decisões de limpeza seguintes.

In [ ]:
# Carregar e inspecionar
df = pd.read_csv("dados/water_potability.csv")

print(f"Shape: {df.shape}")
print(f"\nValores ausentes:")
print(df.isnull().sum())
print(f"\nDuplicatas: {df.duplicated().sum()}")
print(f"\nDistribuição Potability:")
print(df["Potability"].value_counts())

## 3. Imputação de Valores Ausentes

Três colunas apresentam valores ausentes: `ph` (15%), `Sulfate` (23,8%) e
`Trihalomethanes` (4,9%). Remover essas linhas significaria perder até **~35% do dataset**,
reduziria bastante, o que não seria uma coisa boa, então vamos tratar isso.

**Por que mediana e não média?**  
A média é sensível a outliers — um valor extremo puxa a média para longe do centro real
da distribuição. A mediana, por ser o valor do meio, é robusta a isso e representa melhor
o comportamento típico de cada coluna.

**Por que agrupar por classe (`Potability`)?**  
Água potável e não potável têm perfis químicos distintos. Imputar com a mediana global
misturaria essas distribuições, introduzindo um viés sutil nos dados.
Ao calcular a mediana separadamente para cada classe, preservamos as diferenças
estatísticas que os modelos precisam aprender.

In [ ]:
#Imputar valores ausentes
colunas_nulas = [c for c in df.columns[df.isnull().any()] if c != "Potability"]

for col in colunas_nulas:
    medianas = df.groupby("Potability")[col].transform("median")
    df[col] = df[col].fillna(medianas).fillna(df[col].median())
    print(f"'{col}' → NaN imputados com mediana por classe")

## 4. Correção de Valores Inválidos (pH)

O pH é uma escala físico-química definida no intervalo **[0, 14]**. Qualquer valor fora
desse intervalo é fisicamente impossível e indica erro de medição ou entrada de dados.
Esses valores são substituídos pela mediana do pH válido — sem agrupamento por classe
aqui, pois o critério é físico, não estatístico.

In [ ]:
#Corrigir pH fora de [0, 14]
invalidos = ((df["ph"] < 0) | (df["ph"] > 14)).sum()
if invalidos:
    mediana_ph = df.loc[(df["ph"] >= 0) & (df["ph"] <= 14), "ph"].median()
    df.loc[(df["ph"] < 0) | (df["ph"] > 14), "ph"] = mediana_ph
    print(f"pH inválido: {invalidos} valor(es) corrigido(s)")
else:
    print("Nenhum pH fora do intervalo.")

## 5. Tratamento de Outliers — Winsorização pelo IQR

### O que é o IQR?
O **IQR** (*Interquartile Range*, ou Intervalo Interquartil) é a diferença entre o
terceiro quartil (Q3, 75%) e o primeiro quartil (Q1, 25%) de uma distribuição:

IQR = Q3 - Q1

Ele representa a faixa onde estão os **50% centrais dos dados**, sendo por isso
insensível a valores extremos — diferente do desvio padrão, que é distorcido por outliers.

### Como o IQR detecta outliers?
Os limites inferior e superior são definidos como:

Limite inferior = Q1 - 1,5 × IQR
Limite superior = Q3 + 1,5 × IQR

Qualquer valor fora desses limites é considerado um outlier. O fator **1,5** é uma
convenção estatística clássica proposta por John Tukey (o mesmo dos boxplots) e
funciona bem para distribuições aproximadamente simétricas.

### Por que Winsorização e não remoção?
Existem duas abordagens comuns para tratar outliers:
- **Remover** a linha inteira — simples, mas desperdiça dados
- **Winsorizaçāo** (*clipping*) — substitui o valor extremo pelo próprio limite (Q1−1,5×IQR ou Q3+1,5×IQR), mantendo a linha no dataset

Como cada linha representa uma amostra de água que pode ser escassa em cenários reais,
optamos pela winsorização para **preservar o máximo de informação possível**.

In [ ]:
#Winsorização IQR
features = ["ph", "Hardness", "Solids", "Chloramines", "Sulfate",
            "Conductivity", "Organic_carbon", "Trihalomethanes", "Turbidity"]

for col in features:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"'{col}' → {n_out} outlier(s) winsorizados em [{lower:.2f}, {upper:.2f}]")

## 6. Verificação Final e Exportação

Confirmamos que não restam valores ausentes e salvamos o dataset limpo em um arquivo
separado, preservando o CSV original intacto. O arquivo gerado (`water_potability_limpo.csv`)
será a entrada do próximo notebook (`preprocessing.ipynb`), onde faremos o escalonamento
das features e o balanceamento das classes com SMOTE.

In [ ]:
#Verificação final e salvar
print(f"Shape final: {df.shape}")
print(f"NaN restantes: {df.isnull().sum().sum()}")
print(f"\nDistribuição Potability:")
print(df["Potability"].value_counts())

df.to_csv("dados/water_potability_limpo.csv", index=False)
print("\nSalvo em: dados/water_potability_limpo.csv")